[Reference](https://medium.com/activated-thinker/build-a-self-reviewing-ai-agent-using-langgraph-33516814edeb)
```
Answer → Review → Decide → Rewrite → Review → Stop
```

# Step 0: Install Required Libraries (Google Colab)
```
!pip install langgraph langchain langchain-groq

import os

os.environ["GROQ_API_KEY"] = "Copy your api key"
```

# Step 1: Load the Groq LLaMA Model (The Way That Actually Works)

In [1]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3
)
print(llm.invoke("Say hello in one sentence.").content)

# Step 2: Define the Agent’s Memory (Most Important Concept)

In [2]:
from typing import TypedDict

class AgentState(TypedDict):
    question: str
    answer: str
    review: str
    is_good: bool
    retries: int

# Step 3: Answer Node (AI Gives First Answer)

In [3]:
def answer_node(state):
    prompt = f"""
Answer the following question clearly and accurately.

Question:
{state['question']}
"""
    response = llm.invoke(prompt).content
    return {**state, "answer": response}

# Step 4: Review Node

In [4]:
def review_node(state):
    prompt = f"""
You are a strict reviewer.Check the answer below for:
- clarity
- correctness
- completeness
Answer:
{state['answer']}
Reply strictly as:
PASS: <reason>
or
FAIL: <feedback>
"""
    response = llm.invoke(prompt).content
    return {
        **state,
        "review": response,
        "is_good": response.startswith("PASS")
    }

# Step 5: Rewrite Node (AI Improves Using Feedback)

In [5]:
def rewrite_node(state):
    prompt = f"""
Improve the answer using the reviewer feedback.Original Answer:
{state['answer']}
Feedback:
{state['review']}
"""
    response = llm.invoke(prompt).content
    return {
        **state,
        "answer": response,
        "retries": state["retries"] + 1
    }

# Step 6: Decide Whether to Continue or Stop

In [6]:
from langgraph.graph import END

def should_continue(state):
    if state["is_good"]:
        return END
    if state["retries"] >= 2:
        return END
    return "rewrite"

# Step 7: Build the LangGraph

In [7]:
from langgraph.graph import StateGraph

graph = StateGraph(AgentState)
graph.add_node("answer", answer_node)
graph.add_node("review", review_node)
graph.add_node("rewrite", rewrite_node)
graph.set_entry_point("answer") #start node
graph.add_edge("answer", "review") #create edge between nodes
graph.add_conditional_edges(
    "review",
    should_continue,
    {
        "rewrite": "rewrite",
        END: END
    }
)
graph.add_edge("rewrite", "review")
app = graph.compile()

# Step 8: Run the Agent

In [8]:
initial_state = {
    "question": "Explain LangGraph in simple terms.",
    "answer": "",
    "review": "",
    "is_good": False,
    "retries": 0
}

result = app.invoke(initial_state)
print(result["answer"])